# Telco Customer Churn - Data Cleaning & EDA
This notebook covers Step 2 of the Telco Churn Analysis project, performing data cleaning, feature engineering, exploratory data analysis (EDA), and statistical validation using Chi-Square tests to identify significant churn drivers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data & Initial Inspection

In [ ]:
df = pd.read_csv('Telco_Customer_Churn.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Fix TotalCharges: Contains blank spaces for new customers (tenure = 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle the resulting NaNs by filling them with 0 (since tenure is 0, total charges should be 0)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"Missing values after cleaning:\n{df.isnull().sum().max()}")

In [ ]:
# Feature Engineering: Total Services Count
services = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

df['TotalServices'] = (df[services] == 'Yes').sum(axis=1) + (df['InternetService'] != 'No').astype(int)

# Feature Engineering: Tenure Buckets
bins = [0, 12, 24, 48, 60, 100]
labels = ['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years']
df['TenureBucket'] = pd.cut(df['tenure'], bins=bins, labels=labels, right=False)

# Target Variable: Convert Churn to binary (1/0) for correlation analysis
df['ChurnBinary'] = df['Churn'].map({'Yes': 1, 'No': 0})

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
churn_rate = df['Churn'].value_counts(normalize=True) * 100
print(f"Overall Churn Rate: {churn_rate['Yes']:.2f}%")

sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Churn Distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[0], palette='Set2')
axes[0].set_title('Churn by Contract Type')

sns.countplot(data=df, x='TenureBucket', hue='Churn', ax=axes[1], palette='Set2')
axes[1].set_title('Churn by Tenure Bucket')
axes[1].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[2], palette='Set2')
axes[2].set_title('Churn by Internet Service')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False, palette='Set2')
plt.title('Density of Monthly Charges by Churn')
plt.show()

## 4. Statistical Validation
We will use Chi-Square tests to determine if categorical variables have a statistically significant relationship with Churn.

In [ ]:
def chi_square_test(col):
    contingency_table = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    return p

categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 
                   'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                   'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureBucket']

results = []
for col in categorical_cols:
    if col in df.columns:
        p_val = chi_square_test(col)
        results.append({'Feature': col, 'P-Value': p_val, 'Significant (p < 0.05)': p_val < 0.05})

sig_df = pd.DataFrame(results).sort_values(by='P-Value')
display(sig_df)

## 5. Export Cleaned Dataset
We export the engineered dataset for SQL Analysis and Dashboarding.

In [ ]:
df.to_csv('Telco_Customer_Churn_Cleaned.csv', index=False)
print("Cleaned dataset saved as 'Telco_Customer_Churn_Cleaned.csv'")